# Three-asset integer portfolio optimization

Retain the original covariance, returns, integer holdings in 0..7 and budget 6. Native Python describes the objective; the existing quantum function constructs all three QAOA layers and its bounded slack register. This local demonstration makes optimizer settings and seeds explicit and does not claim hardware advantage or the hosted optimizer trajectory.


In [ ]:
from io import BytesIO

from IPython.display import Image, display
from matplotlib import pyplot as plt

from examples.synthesis.applications.finance.portfolio_optimization.main import (
    DEFAULT_PARAMS,
)
from examples.synthesis.applications.finance.portfolio_optimization.overview.operations import (
    build_program,
    verify,
)
from examples.synthesis.applications.finance.portfolio_optimization.overview.workflow import (
    COVARIANCE,
    RETURNS,
    classical_optima,
    objective,
    plot_costs,
    plot_trace,
    run_workflow,
)


In [ ]:
print("Returns:", RETURNS)
print("Covariance:", COVARIANCE)
print("Integer holdings: three values in 0..7; total budget: 6")


In [ ]:
print("Example feasible holding:", (1, 2, 1), "Original cost:", objective((1, 2, 1)))
print("Feasibility rule: sum(holdings) <= 6")
print("The quantum encoding retains slack weights 1, 2, 3 and penalty 10.")


## Native quantum program

`build_program` binds six real parameters to the frozen native `main` quantum function. It retains three assets, the original slack weights 1/2/3 and three cost/mixer layers. Synthesis checks width, workspace, operation and memory budgets. No source importer, physical wire numbering or bitstring decoding is needed.


In [ ]:
program = build_program(DEFAULT_PARAMS)
print("Named registers:", tuple(program.registers))


In [ ]:
print("Actual resources:", program.report["resources"])
checks = (
    verify(DEFAULT_PARAMS),
    verify((-0.023, 0.041, -0.017, 0.052, -0.033, 0.061)),
)
print("Independent absolute-state checks:", checks)


## Bounded native optimization

Retain quantile 0.7 and a 60-iteration budget. Native COBYLA evaluates the lowest 70 percent of the complete encoded-cost distribution, including a fractional probability at the boundary. Exact local probabilities replace the hosted estimator. The initial schedule and trust radii are explicit. A budgeted run need not converge; report its actual termination and evaluation count.


In [ ]:
workflow = run_workflow(maxiter=60, shots=1000, seed=1914)
print(
    "Iterations:",
    workflow.optimizer.iterations,
    "Evaluations:",
    workflow.optimizer.evaluations,
)
print("Converged:", workflow.optimizer.converged)
print(
    "Initial CVaR:", workflow.trace[0].cvar, "Selected CVaR:", workflow.optimizer.value
)


In [ ]:
trace_figure = plot_trace(workflow)
trace_png = BytesIO()
trace_figure.savefig(trace_png, format="png", dpi=130)
display(Image(trace_png.getvalue()))
plt.close(trace_figure)


## Actual observations and uniform baseline

Draw 1,000 actual local observations from the optimized distribution with seed 1914. The original source leaves the shot count unspecified; this native workflow declares it. The separate uniform baseline uses seed 1915. Slack is marginalized by name. Report both feasibility and original risk-minus-return cost, and include infeasible holdings in the histogram rather than concealing them.


In [ ]:
print("Optimized shots:", sum(row.count for row in workflow.optimized))
print("Top feasible-first observed holdings:")
for row in workflow.optimized[:5]:
    print(row)


In [ ]:
print("Uniform shots:", sum(row.count for row in workflow.uniform))
print(
    "Optimized feasible shots:",
    sum(row.count for row in workflow.optimized if row.feasible),
)
print(
    "Uniform feasible shots:",
    sum(row.count for row in workflow.uniform if row.feasible),
)


In [ ]:
cost_figure = plot_costs(workflow)
cost_png = BytesIO()
cost_figure.savefig(cost_png, format="png", dpi=130)
display(Image(cost_png.getvalue()))
plt.close(cost_figure)


In [ ]:
best = workflow.best_observed
if best is None:
    print("No feasible holding was observed.")
else:
    print(
        "Best observed feasible holding:",
        best.weights,
        "Original cost:",
        best.cost,
        "Count:",
        best.count,
    )


## Independent classical optimum

Enumerate all 512 original holdings and apply the at-most-six budget. This replaces an external Couenne solve with an exact check of the full finite domain. It certifies the classical optimum; it never fills in the quantum observations or implies that QAOA must find an optimal holding.


In [ ]:
optima = classical_optima()
print("Independent classical optima:", optima)
print(
    "All 512 holdings checked; feasible holdings:",
    sum(a + b + c <= 6 for a in range(8) for b in range(8) for c in range(8)),
)
